# V14 Self-Play (Colab)

**Goal:** generate V14 self-play games as the steady-state corpus for the pillar4a distillation step. Crisis mining (run separately on M5) contributes the failure-mode coverage; the historic 4800-sim crisis corpus additionally enters pillar4a training as permanent replay (HISTORY 168).

**Player:** `pillar3f + value_head_pillar3f + q_weight=2.0` — current best (HISTORY 167-168). pillar3f = pillar3b + 0.5·(crisis task vector); 5k eval: **mean 31.6k, P50 22.2k, <1000 1.6%, max 301k**. value_head_pillar3f retrained on pillar3f's backbone per the HISTORY 158 rule (K-rollout r = 0.88/0.90/0.92/0.93 across H=25/50/100/200).

**Settings (V14, carried from the established plan):**
- **400 sims** (V13's budget; the prior is stronger, search refines + Dirichlet diversifies)
- q_weight=2.0 (the value-head calibration — HISTORY 133; the LINEAR evaluator would need 0.5)
- batch_size=8 (HISTORY 115)
- **max-turns=10000** (floor focus; NOTE: pillar3f's median game is ~10.5k turns, so roughly half the games will cap — raise to 15000 if late-game states look underrepresented in the first batch)
- **temperature-moves=10** (V14 plan: RNG provides spawn diversity; V13's 15 looked over-noisy early)
- Dirichlet α=0.3, weight=0.25 (defaults)

**Canary protocol:** run a small first shard (~20 seeds), check per-game scores land in/above pillar3f's policy-only ballpark (~22k median uncapped; capped games print `[CAPPED]`). If games come in far below, the head/q is misleading the search — stop and q-sweep before the long campaign.

## Upload to Drive (`MyDrive/alphatrain/`)

| file | size | source |
|---|---|---|
| `colorlines_pillar3d_v2.tar.gz` | ~450 KB | current code tarball (already on Drive) |
| `pillar3f.pt` | ~45 MB | local: `alphatrain/data/pillar3f.pt` |
| `value_head_pillar3f.pt` | ~38 KB | local: `alphatrain/data/value_head_pillar3f.pt` |

## Seed ranges (disjoint from existing data)

- V13 selfplay used 1300000-1300999
- V13 crisis used 800000-808000 (M5)
- **V14 selfplay**: 1500000+ (this notebook; shard across sessions/machines by sub-range)
- V14 crisis (M5, separate): 1600000+

## After this notebook

1. M5 generates V14 crisis in parallel (see Notes cell)
2. Build V14 tensor: `build_expert_v2_tensor --games-dir data/selfplay_v14 data/crisis_v14 --policy-only-data --output alphatrain/data/v14_pillar3f.pt`
3. Train pillar4a: warm-start **pillar3f**, target-temperature 0.5, lr 3e-4, **+ the full 63.8k crisis corpus as ~5% permanent replay per batch** (retention — 400-sim search must not wash out the 4800-sim corrections; HISTORY 168)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/pillar3f.pt /content/alphatrain/data/pillar3f.pt
!cp {DRIVE}/value_head_pillar3f.pt /content/alphatrain/data/value_head_pillar3f.pt
print(f'Model: {os.path.getsize("/content/alphatrain/data/pillar3f.pt")/1e6:.0f} MB')
print(f'Value head: {os.path.getsize("/content/alphatrain/data/value_head_pillar3f.pt")/1e3:.0f} KB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

In [ ]:
# === V14 Self-Play ===
# Shard across Colab instances / the M5 by editing SEED_START/SEED_END.
# First session: run a SMALL canary shard (e.g. 1500000-1500020), check scores,
# then widen. V14 crisis (M5 local) uses 1600000+ — ranges stay disjoint.
SEED_START = 1500000
SEED_END = 1500020       # canary first; widen to e.g. 1500200 after
# =======================

SIMS = 400               # same as V13
Q_WEIGHT = 2.0           # value-head calibration (HISTORY 133); linear would be 0.5
WORKERS = 24             # Colab CPU
BATCH_SIZE = 8           # HISTORY 115
MAX_TURNS = 10000        # floor focus; ~half of pillar3f games will cap — see header note
TEMP_MOVES = 10          # V14 plan: RNG provides spawn diversity
SAVE_DIR = f'{DRIVE}/selfplay_v14'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} games)')
print(f'Sims: {SIMS}, q_weight: {Q_WEIGHT}, cap: {MAX_TURNS} turns, temp_moves: {TEMP_MOVES}')
print(f'Workers: {WORKERS}, bs: {BATCH_SIZE}')
print(f'Save: {SAVE_DIR}')

!python -m alphatrain.scripts.selfplay \
    --model /content/alphatrain/data/pillar3f.pt \
    --value-head-path /content/alphatrain/data/value_head_pillar3f.pt \
    --q-weight {Q_WEIGHT} \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR} \
    --workers {WORKERS} --device cuda \
    --temperature-moves {TEMP_MOVES} \
    --max-turns {MAX_TURNS} \
    --compile

## Notes

- **Resume support:** if Colab disconnects, just re-run the previous cell. It skips seeds whose game files already exist in `SAVE_DIR` (which lives on Drive, so progress survives the VM).
- **Wall time:** pillar3f at MCTS@400 with cap=10K — stronger prior means longer games than V13's; expect ~3-6 min/game in 24-worker parallel. If a shard risks the 24h kill, split the seed range across instances.
- **Watch:** GPU server prints `[GPU] X evals, Y fwd (avg bs=Z, ZZZZ evals/s)` every 10K forwards. avg bs should be ≥50 and evals/s in the thousands. If avg bs drops below 30, workers are starving — usually fine but worth flagging.
- **Output:** each game saved as `game_seed{N}_score{S}.json` in the save dir.
- **Crisis mining (M5 in parallel, seeds 1600000+):**
  ```
  caffeinate -dim python -m alphatrain.scripts.crisis_mining \
      --model alphatrain/data/pillar3f.pt \
      --value-head-path alphatrain/data/value_head_pillar3f.pt \
      --q-weight 2.0 \
      --seed-start 1600000 --seed-end 1601000 \
      --recovery-turns 15 --recovery-sims 600 \
      --prevention-turns 30 --prevention-sims 400 \
      --continue-turns 500 \
      --policy-max-turns 12000 \
      --workers 16 --device mps --batch-size 8 \
      --max-turns 25000 \
      --save-dir data/crisis_v14
  ```

## After both run

Build V14 tensor (M5, ~30 min):
```
python -m alphatrain.scripts.build_expert_v2_tensor \
    --games-dir data/selfplay_v14 data/crisis_v14 \
    --policy-only-data \
    --output alphatrain/data/v14_pillar3f.pt
```

Then train pillar4a (Colab, ~12h) — warm-start **pillar3f**, plus the crisis-replay retention term (the historic 4800-sim corpus must not be washed out by 400-sim targets; HISTORY 168):
```
python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v14_pillar3f.pt --amp --compile \
    --resume alphatrain/data/pillar3f.pt --warm-start \
    --epochs 17 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --target-temperature 0.5 \
    --aux-corrections-corpus crisis/corrections_corpus.pt --aux-weighted \
    --aux-batch-size 1638 --aux-lambda 0.05 --aux-target-temperature 0.5 \
    --aux-holdout-frac 0.15 --aux-split-seed 0
```
(Replay sizing ≈5% of each batch; λ for the replay term to be grad-audited before the run — the exact retention weighting is the one open knob. If pillar4a regresses on held-out crisis match vs pillar3f, raise it; the task-arithmetic channel remains available to re-inject post-hoc.)

Decision criterion: pillar4a ≥ +15-20% over pillar3f mean OR floor cut ≥30% — and held-out crisis match must NOT drop below pillar3f's (retention check). Eval per epoch on the canonical 1k (775000-775999, eval_policy, batch 256) vs pillar3f's 22.2k/31.6k 5k bar.